# Can we predict late delivery?
Protocol: `protocol.yaml`. Temporal holdout (`holdout.csv`), walk-forward CV (5 splits) on training rows, selection by mean CV ROC AUC, holdout scored once.

Feature note: `claim_paid_eur` is recorded after delivery (a claim settled for the consignment), so it is not known at prediction time and is excluded. `consignment` (identifier) and `dispatched_on` (split only) are excluded per protocol.

In [ ]:
import hashlib, json, numpy as np, pandas as pd, yaml
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
from sklearn.base import clone
proto=yaml.safe_load(open('protocol.yaml'))
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()==proto['dataset']['sha256']
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()==proto['split']['holdout_sha256']
df=pd.read_csv('data.csv',parse_dates=['dispatched_on'])
assert len(df)==proto['dataset']['rows']
hold_idx=pd.read_csv('holdout.csv')['row'].values
assert len(hold_idx)==proto['split']['holdout_rows']
train_mask=~df.index.isin(hold_idx)
train=df[train_mask].sort_values('dispatched_on',kind='stable')
hold=df.loc[hold_idx]
print(len(train),len(hold),train.dispatched_on.max(),hold.dispatched_on.min())
print(train.late_delivery.mean(),hold.late_delivery.mean())

In [ ]:
num=['distance_km','weight_kg','parcels']; cat=['carrier']
features=num+cat
def pre(): return ColumnTransformer([('n',Pipeline([('i',SimpleImputer(strategy='median')),('s',StandardScaler())]),num),
                                    ('c',OneHotEncoder(handle_unknown='ignore'),cat)])
# all candidates named before any scoring
candidates={
 'baseline_prior':Pipeline([('p',pre()),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':Pipeline([('p',pre()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':Pipeline([('p',pre()),('m',LogisticRegression(C=1,max_iter=1000))]),
 'random_forest':Pipeline([('p',pre()),('m',RandomForestClassifier(300,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hist_gb':Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
X=train[features]; y=train.late_delivery.values
tscv=TimeSeriesSplit(n_splits=proto['cv']['n_splits'])
folds=[(tr,va) for tr,va in tscv.split(X)]
cv_scores={}
for name,est in candidates.items():
    cv_scores[name]=[roc_auc_score(y[va],clone(est).fit(X.iloc[tr],y[tr]).predict_proba(X.iloc[va])[:,1]) for tr,va in folds]
res=pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T
print(res.sort_values('mean',ascending=False))
chosen=res['mean'].idxmax(); print('chosen:',chosen)

In [ ]:
final=clone(candidates[chosen]).fit(X,y)
hold_auc=roc_auc_score(hold.late_delivery,final.predict_proba(hold[features])[:,1])
print('Holdout ROC AUC (scored once):',hold_auc)
pos=lambda idx:[int(i) for i in train.index[idx]]
evidence={'features':features,'candidates':list(candidates),
 'cv_folds':[[pos(tr),pos(va)] for tr,va in folds],
 'cv_scores':cv_scores,'chosen':chosen,
 'holdout_rows':[int(i) for i in hold_idx],
 'holdout_scores':{chosen:hold_auc}}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the CV table and the holdout ROC AUC above. The holdout score is the headline; nothing was selected on it.